# Comparison of feast networks
- individual Sundays of Advent
- st. Mary feasts (Annunciatio, Purificatio, )

Building blocks

In [4]:
import pycantus
import pycantus.data as data
from pycantus.filtration import Filter
import copy
import utils
import importlib

In [5]:
corpus = data.load_dataset('cantuscorpus_v1.0')

Loading chants and sources...
Data loaded!


In [6]:
n_chants = len(corpus.chants)
print(f'Number of chants in CantusCorpus v1.0: {n_chants}')
n_sources = len(corpus.sources)
print(f'Number of sources in CantusCorpus v1.0: {n_sources}')

Number of chants in CantusCorpus v1.0: 888010
Number of sources in CantusCorpus v1.0: 2278


In [7]:
# Clean the corpus
# Drop doxology
doxo_filter = pycantus.filtration.Filter('doxo_filter')
doxo_filter.add_value_exclude('cantus_id', '909000')
corpus.apply_filter(doxo_filter)

# Drop fragments => sources with less than 100 chants
corpus.drop_small_sources_data(min_chants=100)

In [8]:
n_chants = len(corpus.chants)
print(f'Number of chants in CantusCorpus v1.0: {n_chants}')
n_sources = len(corpus.sources)
print(f'Number of sources in CantusCorpus v1.0: {n_sources}')

Number of chants in CantusCorpus v1.0: 855146
Number of sources in CantusCorpus v1.0: 510


### Advent

In [9]:
# Build corpora for Sundays, e.g. "Dom. 1 Adventus"
advent_sundays_corpora = {}
for i in range(1, 5):
    feast = f'Dom. {i} Adventus'
    sunday_corpus = copy.deepcopy(corpus)
    sunday_filter = Filter(f'{feast}_filter')
    sunday_filter.add_value_include('feast', feast)
    sunday_corpus.apply_filter(sunday_filter)
    sunday_corpus.drop_empty_sources()
    advent_sundays_corpora[feast] = sunday_corpus
    print(f'{feast}: {len(sunday_corpus.chants)} chants, {len(sunday_corpus.sources)} sources')

Dom. 1 Adventus: 6447 chants, 245 sources
Dom. 2 Adventus: 4544 chants, 240 sources
Dom. 3 Adventus: 4876 chants, 251 sources
Dom. 4 Adventus: 4689 chants, 251 sources


In [13]:
# Bulit copora for advent sundays Lauds only
advent_sundays_lauds_corpora = {}
lauds_filter = Filter(f'lauds_filter')
for i in range(1, 5):
    feast = f'Dom. {i} Adventus'
    sunday_corpus = advent_sundays_corpora[feast].copy()
    lauds_filter.add_value_include('office', 'L')
    sunday_corpus.apply_filter(lauds_filter)
    sunday_corpus.drop_empty_sources()
    advent_sundays_lauds_corpora[feast] = sunday_corpus
    print(f'{feast} Lauds: {len(sunday_corpus.chants)} chants, {len(sunday_corpus.sources)} sources')

AttributeError: 'Corpus' object has no attribute 'copy'

### Networks

In [10]:
import networkx as nx

In [11]:
advent_sundays_nets = {}
for i in range(1, 5):
    feast = f'Dom. {i} Adventus'
    sunday_corpus = advent_sundays_corpora[feast]
    G = utils.build_feast_network(sunday_corpus, feast)
    advent_sundays_nets[feast] = G
    print(f'{feast}: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
    # Save the network
    nx.write_edgelist(G, f'nets/{feast}_network.edgelist')

Dom. 1 Adventus: 245 nodes, 13735 edges
Dom. 2 Adventus: 240 nodes, 13306 edges
Dom. 3 Adventus: 251 nodes, 13927 edges
Dom. 4 Adventus: 251 nodes, 14066 edges


In [12]:
importlib.reload(utils)

for G in advent_sundays_nets.values():
    utils.graph_info_nx(G, fast=False)
    print('--------------------------------------')

       Graph | 'Dom. 1 Adventus'
       Nodes | 245 (3)
       Edges | 13,735 (0)
      Degree | 112.12 (227)
  Components | 97.6% (5)
  Clustering | 0.9426
Modularity from Louvain | 0.4286 (6)

--------------------------------------
       Graph | 'Dom. 2 Adventus'
       Nodes | 240 (0)
       Edges | 13,306 (0)
      Degree | 110.88 (230)
  Components | 98.8% (2)
  Clustering | 0.9752
Modularity from Louvain | 0.4121 (4)

--------------------------------------
       Graph | 'Dom. 3 Adventus'
       Nodes | 251 (1)
       Edges | 13,927 (0)
      Degree | 110.97 (233)
  Components | 97.2% (4)
  Clustering | 0.9603
Modularity from Louvain | 0.4415 (7)

--------------------------------------
       Graph | 'Dom. 4 Adventus'
       Nodes | 251 (0)
       Edges | 14,066 (0)
      Degree | 112.08 (237)
  Components | 98.4% (2)
  Clustering | 0.9652
Modularity from Louvain | 0.4952 (4)

--------------------------------------
